# PushT sanity check (Colab)

Thin notebook to test-run the latency-robust chunking sweep with the pretrained
`lerobot/diffusion_pusht` checkpoint. **Set the runtime to GPU** (Runtime → Change runtime type
→ T4 GPU).

Three escalating checks:
1. **Core logic** — pure-numpy, no GPU; confirms the scheduler + strategies discriminate.
2. **Policy smoke** — load the real policy, run one PushT episode (shakes out version-sensitive lines).
3. **Mini sweep** — latency × strategy success curve, plotted inline.

The real contribution is the ROS edge-network testbed in the repo; this just validates the
phenomenon cheaply first.

In [ ]:
!nvidia-smi -L  # confirm a GPU is attached

In [1]:
# Install LOCKED deps via the pusht extra (auto-resolves gym-pusht + pymunk + gymnasium 0.29.x).
# lerobot 0.4+ moved normalization out of the policy (breaks from_pretrained + select_action for
# this checkpoint); 0.3.3 is the last release where it 'just works'.
!pip install -q "lerobot[pusht]==0.3.3" matplotlib
# lerobot 0.3.3 needs torch>=2.2.1,<2.8.0. If pip downgrades Colab's torch and CUDA breaks:
# restart runtime, then `!pip install "torch==2.7.*"` before re-running this cell.

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 597.7/597.7 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 953.9/953.9 kB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.4/51.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.0/821.0 MB 2.0 MB/s eta 0

In [2]:
# Get the rig code (force-sync to remote so stale local edits don't linger) + headless rendering.
import os, sys, subprocess

REPO = 'https://github.com/HenryNVP/edge-vla-hil.git'
BRANCH = 'main'
if not os.path.exists('edge-vla-hil'):
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO], check=True)
else:
    subprocess.run(['git', '-C', 'edge-vla-hil', 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', 'edge-vla-hil', 'reset', '--hard', f'origin/{BRANCH}'], check=True)
sys.path.insert(0, 'edge-vla-hil/experiments/pusht_sanity')

os.environ['SDL_VIDEODRIVER'] = 'dummy'   # headless pygame
os.environ['MUJOCO_GL'] = 'egl'

# Diagnostic: show installed lerobot version + top-level layout (which namespace exists).
import lerobot, pkgutil
print('lerobot', getattr(lerobot, '__version__', '?'),
      '| submodules:', [m.name for m in pkgutil.iter_modules(lerobot.__path__)])
print('rig path ready')

lerobot 0.3.3 | submodules: ['__version__', 'calibrate', 'cameras', 'constants', 'envs', 'errors', 'find_cameras', 'find_port', 'motors', 'optim', 'policies', 'processor', 'record', 'replay', 'robots', 'setup_motors', 'teleoperate', 'teleoperators']
rig path ready


## 1. Core logic (no GPU) — does the harness discriminate?

Expected, matching the RTC paper: as latency grows, `naive_async` loses success / balloons steps,
`synchronous` keeps success but pays throughput, `rtc_freeze` stays most robust.

In [3]:
import numpy as np
from latency_chunking import (
    run_episode, make_strategy, DelayModel, PointMassEnv, make_mock_policy)

print('toy task: success%% / mean-steps over 30 seeds, jitter=2')
for lat in [0, 4, 8, 12]:
    cells = []
    for name in ['synchronous', 'naive_async', 'temporal_ensemble', 'rtc_freeze']:
        rs = [run_episode(PointMassEnv(), make_mock_policy(), make_strategy(name),
                          DelayModel(latency_steps=lat, jitter_steps=2, seed=k), max_steps=200)
              for k in range(30)]
        sr = np.mean([r.success for r in rs]) * 100
        st = np.mean([r.steps for r in rs])
        cells.append(f'{name[:5]}:{sr:3.0f}%/{st:4.0f}')
    print(f'lat={lat:2d}  ' + '  '.join(cells))

toy task: success%% / mean-steps over 30 seeds, jitter=2
lat= 0  synch:100%/  36  naive:100%/  23  tempo:100%/  22  rtc_f:100%/  20
lat= 4  synch:100%/  55  naive:100%/  44  tempo:100%/  32  rtc_f:100%/  31
lat= 8  synch:100%/  61  naive: 97%/ 103  tempo:100%/  34  rtc_f:100%/  32
lat=12  synch:100%/  70  naive: 80%/ 151  tempo:100%/  34  rtc_f:100%/  34


## 2. Policy smoke — load pretrained diffusion_pusht, run one episode

First run downloads the checkpoint. If this cell errors, it's almost always one of the 3 lines
marked `###` in `run_pusht.py` (policy import / action-chunk call / obs batch) vs. your installed
`lerobot` version.

In [12]:
from run_pusht import load_policy, GymPushTEnv
from latency_chunking import run_episode, make_strategy, DelayModel

policy_fn, action_dim, horizon = load_policy('cuda')
print('loaded policy; horizon =', horizon)

# Create the environment instance
env = GymPushTEnv()
# Access the first standard wrapper and then use .unwrapped to get to the base environment
env.env.unwrapped.render_mode = 'rgb_array'

r = run_episode(env, policy_fn, make_strategy('rtc_freeze'),
                DelayModel(latency_steps=0), max_steps=300)
print(r)

loaded policy; horizon = 8
EpisodeResult(steps=300, total_reward=97.38711881661384, max_reward=0.9411314985950785, success=False, paused_steps=0, replans=38)


## 3. Mini sweep + inline plot

Small (10 episodes/cell) so it finishes in a few minutes on a T4. Bump `EPS` / `lats` for a real
curve. The headline you want: the strategy lines separating as latency grows.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from run_pusht import load_policy, GymPushTEnv
from latency_chunking import run_episode, make_strategy, DelayModel

try:
    policy_fn
except NameError:
    policy_fn, _, _ = load_policy('cuda')

strategies = ['synchronous', 'naive_async', 'temporal_ensemble', 'rtc_freeze']
lats = [0, 2, 4, 8]
EPS = 10

curves = {s: [] for s in strategies}
for s in strategies:
    for lat in lats:
        # Create and configure environments for each episode
        configured_envs = []
        for _ in range(EPS):
            env_instance = GymPushTEnv()
            # Access the base environment within the gymnasium wrappers and set its render_mode
            env_instance.env.unwrapped.render_mode = 'rgb_array'
            configured_envs.append(env_instance)

        succ = [run_episode(configured_envs[e], policy_fn, make_strategy(s),
                            DelayModel(latency_steps=lat, jitter_steps=1, seed=e),
                            max_steps=300).success
                for e in range(EPS)]
        curves[s].append(float(np.mean(succ)))
        print(f'{s:18s} lat={lat:2d}  success={np.mean(succ)*100:5.1f}%')

for s in strategies:
    plt.plot(lats, curves[s], marker='o', label=s)
plt.xlabel('injected latency (control steps)')
plt.ylabel('success rate')
plt.ylim(0, 1.02)
plt.title('PushT: latency-robust chunking strategies')
plt.legend(); plt.grid(alpha=0.3); plt.show()

synchronous        lat= 0  success= 60.0%
synchronous        lat= 2  success= 60.0%


## Next

- If the curves separate as expected, the phenomenon + harness are validated — move to the
  robosuite/RoboMimic headline (wire `evh_plant` + the ONNX backend).
- For a publishable-quality PushT curve: `EPS=50`, `lats=[0,2,4,8,12,16]`, add `--jitter` and a
  `drop_prob` sweep (pass `drop_prob=` to `DelayModel`).
- Headless errors on `GymPushTEnv()`? Re-run the setup cell (`SDL_VIDEODRIVER=dummy`).